# Notebook E: Q&A Instruction Fine-Tuning

**Goal**: Fine-tune a QLoRA adapter on top of the Phase 1 merged base model using the Unsloth framework and TRL `SFTTrainer` on Kaggle (T4 GPU).

## 0. Install

In [4]:

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers "trl" peft accelerate bitsandbytes datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Environment Detection & Device Setup

In [5]:
import os
import sys

# Force single GPU visibility to prevent DDP multi-GPU configuration crashes on Kaggle T4 x2
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Set CUDA_VISIBLE_DEVICES to '0'")

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_GPU" in os.environ

if IS_KAGGLE:
    MODEL_NAME = "/kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf"
    TRAIN_DS_PATH = "/kaggle/input/datasets/rafaelvieira1/spurgeon-qa-dataset-chatml-template/qa_dataset_train"
    VAL_DS_PATH = "/kaggle/input/datasets/rafaelvieira1/spurgeon-qa-dataset-chatml-template/qa_dataset_val"
    OUTPUT_DIR = "/kaggle/working/qa_checkpoints"
    FINAL_ADAPTER_DIR = "/kaggle/working/spurgeon_lora_qa"
elif IS_COLAB:
    MODEL_NAME = "/content/spurgeon_phase1_merged"
    TRAIN_DS_PATH = "/content/qa_dataset_train"
    VAL_DS_PATH = "/content/qa_dataset_val"
    OUTPUT_DIR = "/content/qa_checkpoints"
    FINAL_ADAPTER_DIR = "/content/spurgeon_lora_qa"
else:
    MODEL_NAME = "../models/spurgeon_phase1_merged"
    TRAIN_DS_PATH = "../data/qa_dataset_train"
    VAL_DS_PATH = "../data/qa_dataset_val"
    OUTPUT_DIR = "../models/qa_checkpoints"
    FINAL_ADAPTER_DIR = "../models/spurgeon_lora_qa"

print(f"Base Model: {MODEL_NAME}")
print(f"Train dataset: {TRAIN_DS_PATH}")

Set CUDA_VISIBLE_DEVICES to '0'
Base Model: /kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf
Train dataset: /kaggle/input/datasets/rafaelvieira1/spurgeon-qa-dataset-chatml-template/qa_dataset_train


## 2. Load Model & Setup LoRA Adapter

In [6]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LENGTH = 2048
LORA_RANK = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Apply the correct ChatML template for Qwen2.5
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# Add PEFT adapter matrices
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,       # Set to 0 to enable Unsloth custom CUDA kernel optimizations
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model and LoRA layers successfully prepared.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The tokenizer you are loading from '/kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth 2026.6.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model and LoRA layers successfully prepared.


## 3. Load Prepared Datasets

In [7]:
import shutil
from datasets import load_from_disk

# Kaggle read-only mapping fallback: Copy dataset to working directory before mapping tokenization caches
LOCAL_TRAIN_PATH = "/kaggle/working/qa_dataset_train_local" if IS_KAGGLE else TRAIN_DS_PATH
LOCAL_VAL_PATH = "/kaggle/working/qa_dataset_val_local" if IS_KAGGLE else VAL_DS_PATH

if IS_KAGGLE:
    if os.path.exists(LOCAL_TRAIN_PATH):
        shutil.rmtree(LOCAL_TRAIN_PATH)
    shutil.copytree(TRAIN_DS_PATH, LOCAL_TRAIN_PATH)
    if os.path.exists(LOCAL_VAL_PATH):
        shutil.rmtree(LOCAL_VAL_PATH)
    shutil.copytree(VAL_DS_PATH, LOCAL_VAL_PATH)
    print("Copied datasets to local workspace for read-write compatibility.")

train_ds = load_from_disk(LOCAL_TRAIN_PATH)
val_ds = load_from_disk(LOCAL_VAL_PATH)

print(f"Loaded {len(train_ds):,} train examples and {len(val_ds):,} val examples.")

Copied datasets to local workspace for read-write compatibility.
Loaded 2,647 train examples and 140 val examples.


## 4. Training Arguments & SFT Config

In [8]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # Effective batch size = 16
    num_train_epochs=3,                 # 3 epochs is sufficient for SFT
    warmup_steps=50,
    learning_rate=1e-4,                 # Fine-tuning rate (lower than Phase 1 pre-train)
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,                      # IMPORTANT: Do not pack Q&A pairs together to keep distinct boundary contexts
    seed=42,
)

## 5. Initialize Trainer & Apply Serialization Fix

In [9]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

# Apply SFTConfig serialization/pickling crash fix (modules mismatch workaround)
try:
    import sys
    if 'trl.trainer.sft_config' in sys.modules:
        sys.modules['trl.trainer.sft_config'].SFTConfig = trainer.args.__class__
        print("Successfully bound SFTConfig serialization hook.")
except Exception as e:
    print(f"Warning binding serialization hook: {e}")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2647 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/140 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Successfully bound SFTConfig serialization hook.


## 6. Execute Training

In [10]:
# Execute training run
trainer_stats = trainer.train()

print(f"Training finished. Time: {trainer_stats.metrics['train_runtime']:.2f}s")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,647 | Num Epochs = 3 | Total steps = 498
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## 7. Save PEFT Adapter

In [ ]:
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Successfully saved LoRA adapter weights to {FINAL_ADAPTER_DIR}")